In [1]:
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

from src.datasets import FaceDataset, train_transform, val_transform, FaceTestDataset
from src.model import MyCNN
from src.engine import train_one_epoch, evaluate
from src.utils import find_best_threshold

In [2]:
DATA_DIR = Path("dataset")
TRAIN_DIR = DATA_DIR / "train_images"
TRAIN_CSV = DATA_DIR / "train_solution.csv"
TEST_DIR = DATA_DIR / "test_images"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

cuda
NVIDIA GeForce RTX 3080


In [3]:
df = pd.read_csv(TRAIN_CSV, header=None, names=["Id", "target"])
df["path"] = df["Id"].apply(lambda x: TRAIN_DIR / f"{x}.jpg")

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["target"]
)

train_df = train_df.sample(min(1000, len(train_df)), random_state=42)
val_df = val_df.sample(min(200, len(val_df)), random_state=42)

print(len(train_df), len(val_df))

1000 200


In [4]:
train_dataset = FaceDataset(train_df, transform=train_transform)
val_dataset = FaceDataset(val_df, transform=val_transform)

torch.backends.cudnn.benchmark = True

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

In [5]:
model = MyCNN().to(device)

num_pos = (train_df["target"] == 1).sum()
num_neg = (train_df["target"] == 0).sum()
pos_weight = torch.tensor([num_neg / num_pos], dtype=torch.float32, device=device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

In [6]:
import copy
from tqdm.notebook import tqdm
best_val_f1 = -1
best_threshold = 0.5
best_state = None

n_epochs = 20

for epoch in range(n_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_f1_05, val_probs, val_labels = evaluate(
        model, val_loader, criterion, device, threshold=0.5
    )

    thr, val_f1_best = find_best_threshold(val_labels, val_probs)

    tqdm.write(
        f"Epoch {epoch+1}/{n_epochs} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_f1@0.5={val_f1_05:.4f} | "
        f"best_thr={thr:.2f} | "
        f"best_val_f1={val_f1_best:.4f}"
    )

    if val_f1_best > best_val_f1:
        best_val_f1 = val_f1_best
        best_threshold = thr
        best_state = copy.deepcopy(model.state_dict())

    scheduler.step(val_f1_best)

Epoch 1/20 | train_loss=1.6291 | val_loss=1.2855 | val_f1@0.5=0.3740 | best_thr=0.54 | best_val_f1=0.3866


Epoch 2/20 | train_loss=1.2787 | val_loss=1.4411 | val_f1@0.5=0.2955 | best_thr=0.32 | best_val_f1=0.4503


Epoch 3/20 | train_loss=1.2375 | val_loss=1.3959 | val_f1@0.5=0.3377 | best_thr=0.35 | best_val_f1=0.4211


Epoch 4/20 | train_loss=1.1383 | val_loss=1.4351 | val_f1@0.5=0.2647 | best_thr=0.22 | best_val_f1=0.4182


Epoch 5/20 | train_loss=1.1150 | val_loss=1.3663 | val_f1@0.5=0.3133 | best_thr=0.44 | best_val_f1=0.4561


...

Epoch 16/20 | train_loss=1.0831 | val_loss=1.1650 | val_f1@0.5=0.4672 | best_thr=0.47 | best_val_f1=0.4722


Epoch 17/20 | train_loss=1.0297 | val_loss=1.2395 | val_f1@0.5=0.4490 | best_thr=0.48 | best_val_f1=0.4906


Epoch 18/20 | train_loss=1.0086 | val_loss=1.2036 | val_f1@0.5=0.4364 | best_thr=0.53 | best_val_f1=0.4848


Epoch 19/20 | train_loss=0.9753 | val_loss=1.1675 | val_f1@0.5=0.4255 | best_thr=0.60 | best_val_f1=0.4948


Epoch 20/20 | train_loss=0.9770 | val_loss=1.2233 | val_f1@0.5=0.4348 | best_thr=0.41 | best_val_f1=0.4667


In [7]:
model.load_state_dict(best_state)
print("Best F1:", best_val_f1)
print("Best threshold:", best_threshold)

Best F1: 0.5106382978723404
Best threshold: 0.42000000000000004


In [8]:
from torchvision import transforms
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    ),
])

In [9]:
test_dataset = FaceTestDataset(TEST_DIR, transform=test_transform)
test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4
)

print("Test size:", len(test_dataset))

Test size: 10000


In [10]:
from tqdm import tqdm

@torch.no_grad()
def predict_test(model, loader, device, threshold):
    model.eval()

    ids = []
    preds = []

    for images, image_ids in tqdm(loader, desc="Predicting"):
        images = images.to(device)

        logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy().ravel()
        batch_preds = (probs >= threshold).astype(int)

        ids.extend(image_ids.numpy().tolist())
        preds.extend(batch_preds.tolist())

    return ids, preds

In [11]:
test_ids, test_preds = predict_test(
    model=model,
    loader=test_loader,
    device=device,
    threshold=best_threshold
)

print(test_ids[:10])
print(test_preds[:10])
print(len(test_ids), len(test_preds))

Predicting: 100%|██████████| 157/157 [00:59<00:00,  2.65it/s]

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[0, 0, 1, 0, 0, 0, 0, 1, 1, 0]
10000 10000


In [13]:
submission = pd.DataFrame({
    "id": test_ids,
    "target_feature": test_preds
}).sort_values("id").reset_index(drop=True)

submission.head()
submission.to_csv("submission.csv", index=False)
print("submission.csv saved")

submission.csv saved
